# OCR Manual - Segmentacion por proyeccion + template matching

Sin librerias de OCR. Solo cv2 + numpy.


In [ ]:
import cv2
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from collections import defaultdict
import random


In [ ]:
DATA_DIR = Path('../data')
N_SAMPLES = 100

all_txt = sorted(DATA_DIR.glob('*.txt'))
print(f'Found {len(all_txt)} files')

selected = random.sample(all_txt, min(N_SAMPLES, len(all_txt)))
print(f'Selected {len(selected)} files')


In [ ]:
def preprocess_plate(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray = cv2.medianBlur(gray, 3)
    min_val, max_val = np.min(gray), np.max(gray)
    if max_val > min_val:
        gray = ((gray - min_val) / (max_val - min_val) * 255).astype(np.uint8)

    gray = cv2.resize(gray, None, fx=2.5, fy=2.5, interpolation=cv2.INTER_CUBIC)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)
    return gray


def binarize(gray):
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    if np.sum(binary == 255) < np.sum(binary == 0):
        binary = cv2.bitwise_not(binary)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=1)
    return binary


def smooth_projection(proj, k=9):
    proj = proj.astype(np.float32)
    kernel = np.ones((1, k), dtype=np.float32) / k
    smoothed = cv2.filter2D(proj.reshape(1, -1), -1, kernel).flatten()
    return smoothed


def find_local_valley(proj, left, right, threshold):
    local = proj[left:right + 1]
    idx = int(np.argmin(local))
    if local[idx] <= threshold:
        return left + idx
    return left + idx


def adjust_segments(segments, proj, expected_count):
    if not segments:
        return segments

    segments = segments[:]
    avg_w = (segments[-1][1] - segments[0][0]) / max(expected_count, 1)
    while len(segments) < expected_count:
        idx, (x1, x2) = max(enumerate(segments), key=lambda s: s[1][1] - s[1][0])
        width = x2 - x1
        if width < max(8, int(avg_w * 0.8)):
            break
        left = x1 + max(int(width * 0.2), 1)
        right = x2 - max(int(width * 0.2), 1)
        if right <= left:
            break
        split = left + int(np.argmin(proj[left:right + 1]))
        if split <= x1 + 1 or split >= x2 - 1:
            break
        segments = segments[:idx] + [(x1, split), (split, x2)] + segments[idx + 1:]

    while len(segments) > expected_count:
        gaps = [(i, segments[i + 1][0] - segments[i][1]) for i in range(len(segments) - 1)]
        i, _ = min(gaps, key=lambda t: t[1])
        segments = segments[:i] + [(segments[i][0], segments[i + 1][1])] + segments[i + 2:]

    return segments


def segment_by_projection(gray, expected_count):
    """
    Segmenta en N chars usando proyeccion vertical + ajuste local.
    """
    binary = binarize(gray)
    proj = np.sum(binary == 255, axis=0)
    proj = smooth_projection(proj, k=9)
    valley_threshold = np.max(proj) * 0.25

    w = gray.shape[1]
    boundaries = [0]
    for i in range(1, expected_count):
        expected = int(i * w / expected_count)
        window = max(int(w * 0.05), 8)
        left = max(0, expected - window)
        right = min(w - 1, expected + window)
        boundary = find_local_valley(proj, left, right, valley_threshold)
        boundaries.append(boundary)
    boundaries.append(w)

    segments = []
    for i in range(len(boundaries) - 1):
        x1, x2 = boundaries[i], boundaries[i + 1]
        if x2 - x1 > 2:
            segments.append((x1, x2))

    segments = adjust_segments(segments, proj, expected_count)
    return segments, binary, proj


def extract_char_images(gray, segments):
    binary = binarize(gray)
    chars = []
    h, w = binary.shape
    for x1, x2 in segments:
        roi = binary[:, x1:x2]
        ys, xs = np.where(roi == 255)
        if len(xs) == 0:
            chars.append({'img': roi, 'bbox': (x1, 0, x2 - x1, roi.shape[0])})
            continue
        y1, y2 = ys.min(), ys.max()
        x1b, x2b = xs.min(), xs.max()
        pad = 1
        y1 = max(0, y1 - pad)
        y2 = min(roi.shape[0] - 1, y2 + pad)
        x1b = max(0, x1b - pad)
        x2b = min(roi.shape[1] - 1, x2b + pad)
        char_img = roi[y1:y2 + 1, x1b:x2b + 1]
        chars.append({'img': char_img, 'bbox': (x1 + x1b, y1, x2b - x1b + 1, y2 - y1 + 1)})
    return chars


def normalize_char(img):
    if img is None or img.size == 0:
        return np.zeros((64, 32), dtype=np.uint8)
    h, w = img.shape
    scale = min(32 / max(w, 1), 64 / max(h, 1))
    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))
    resized = cv2.resize(img, (new_w, new_h))
    canvas = np.zeros((64, 32), dtype=np.uint8)
    x0 = (32 - new_w) // 2
    y0 = (64 - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized
    return canvas


def edge_iou(a, b):
    inter = np.logical_and(a > 0, b > 0).sum()
    union = np.logical_or(a > 0, b > 0).sum()
    if union == 0:
        return 0.0
    return inter / union


def match_character(char_img, ref_chars, threshold=0.2):
    char_norm = normalize_char(char_img)
    char_inv = cv2.bitwise_not(char_norm)
    candidates = [(char_norm, cv2.Canny(char_norm, 40, 120)), (char_inv, cv2.Canny(char_inv, 40, 120))]
    best_match = None
    best_score = threshold
    for char_cand, char_edge in candidates:
        char_f = char_cand.astype(np.float32) / 255.0
        for ref_char, ref_images in ref_chars.items():
            for ref_item in ref_images:
                ref_norm = ref_item['img']
                ref_edge = ref_item['edge']
                ref_f = ref_norm.astype(np.float32) / 255.0
                corr = float(cv2.matchTemplate(char_f, ref_f, cv2.TM_CCOEFF_NORMED)[0][0])
                iou = edge_iou(char_edge, ref_edge)
                score = 0.7 * corr + 0.3 * iou
                if score > best_score:
                    best_score = score
                    best_match = ref_char
    return best_match, best_score


def ocr_plate(crop, gt_len, ref_chars):
    gray = preprocess_plate(crop)
    segments, binary, proj = segment_by_projection(gray, gt_len)
    chars = extract_char_images(gray, segments)

    recognized = []
    for char_data in chars:
        matched, score = match_character(char_data['img'], ref_chars)
        if matched is None:
            matched = '?'
            score = 0.0
        recognized.append({'char': matched, 'score': score, 'img': char_data['img'], 'bbox': char_data['bbox']})

    text = ''.join([r['char'] for r in recognized])
    return text, recognized, gray, binary, segments


In [ ]:
print('Building reference database...')
ref_chars = defaultdict(list)
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))

for txt_file in sorted(all_txt)[:80]:
    with open(txt_file) as f:
        line = f.readline().strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 6:
            continue
        img_name, x, y, w, h, gt = parts[0], int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4]), parts[5]
        img = cv2.imread(str(DATA_DIR / img_name))
        if img is None:
            continue
        crop = img[y:y+h, x:x+w]
        gray = preprocess_plate(crop)
        segments, _, _ = segment_by_projection(gray, len(gt))
        chars = extract_char_images(gray, segments)
        if len(chars) == len(gt):
            for char_data, gt_char in zip(chars, gt):
                base = normalize_char(char_data['img'])
                dil = cv2.dilate(base, kernel, iterations=1)
                ero = cv2.erode(base, kernel, iterations=1)
                for variant in (base, dil, ero):
                    ref_chars[gt_char].append({
                        'img': variant,
                        'edge': cv2.Canny(variant, 40, 120)
                    })

print('Reference characters:')
for char in sorted(ref_chars.keys()):
    print(f"  '{char}': {len(ref_chars[char])} samples")


In [ ]:
print('Processing test samples...')
results = []

for txt_file in selected:
    with open(txt_file) as f:
        line = f.readline().strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 6:
            continue
        img_name, x, y, w, h, gt = parts[0], int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4]), parts[5]
        img = cv2.imread(str(DATA_DIR / img_name))
        if img is None:
            continue
        crop = img[y:y+h, x:x+w]
        ocr_text, chars_data, gray, binary, segments = ocr_plate(crop, len(gt), ref_chars)
        match = ocr_text == gt
        avg_score = float(np.mean([c['score'] for c in chars_data])) if chars_data else 0.0

        results.append({
            'image': img_name,
            'gt': gt,
            'ocr': ocr_text,
            'match': match,
            'avg_score': avg_score,
            'chars_data': chars_data,
            'img_original': img,
            'crop': crop,
            'gray': gray,
            'binary': binary,
            'segments': segments,
            'x': x, 'y': y, 'w': w, 'h': h
        })
        status = '✓' if match else '✗'
        print(f"{status} {img_name}: GT={gt:15} OCR={ocr_text:15} (score: {avg_score:.2f})")

print(f"\nTotal procesados: {len(results)}")


In [ ]:
# Visualizar resultados con caracteres
for result in results[:8]:
    img_original = result['img_original']
    crop = result['crop']
    gray = result['gray']
    binary = result['binary']
    chars_data = result['chars_data']
    x, y, w, h = result['x'], result['y'], result['w'], result['h']

    if not chars_data:
        print(f"Skip {result['image']}: no caracteres detectados")
        continue

    # Original con bbox
    bbox_img = img_original.copy()
    cv2.rectangle(bbox_img, (x, y), (x+w, y+h), (0, 255, 0), 2)

    fig = plt.figure(figsize=(18, 6))

    ax1 = plt.subplot(2, len(chars_data)+3, 1)
    ax1.imshow(cv2.cvtColor(bbox_img, cv2.COLOR_BGR2RGB))
    ax1.set_title('Original')
    ax1.axis('off')

    ax2 = plt.subplot(2, len(chars_data)+3, 2)
    ax2.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    ax2.set_title('Crop')
    ax2.axis('off')

    ax3 = plt.subplot(2, len(chars_data)+3, 3)
    ax3.imshow(binary, cmap='gray')
    ax3.set_title('Binary')
    ax3.axis('off')

    for i, char_data in enumerate(chars_data):
        ax = plt.subplot(2, len(chars_data)+3, len(chars_data)+3+1+i)
        char_viz = cv2.resize(char_data['img'], (64, 128))
        ax.imshow(char_viz, cmap='gray')
        ax.set_title(f"{char_data['char']}\n({char_data['score']:.2f})", fontsize=10, fontweight='bold')
        ax.axis('off')

    match = result['match']
    color = 'green' if match else 'red'
    axr = plt.subplot(2, len(chars_data)+3, len(chars_data)+3)
    axr.axis('off')
    axr.text(0.1, 0.75, 'GT:', fontsize=12, fontweight='bold')
    axr.text(0.35, 0.75, result['gt'], fontsize=13, fontweight='bold')
    axr.text(0.1, 0.55, 'OCR:', fontsize=12, fontweight='bold')
    axr.text(0.35, 0.55, result['ocr'], fontsize=13, color=color, fontweight='bold')
    axr.text(0.1, 0.35, 'Match:', fontsize=11)
    axr.text(0.35, 0.35, '✓' if match else '✗', fontsize=14, color=color, fontweight='bold')
    axr.text(0.1, 0.15, 'Score:', fontsize=10)
    axr.text(0.35, 0.15, f"{result['avg_score']:.2f}", fontsize=11)

    plt.suptitle(result['image'], fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
if results:
    matches = sum(1 for r in results if r['match'])
    accuracy = matches / len(results) * 100
    avg_score = np.mean([r['avg_score'] for r in results])
    print('\nRESULTADOS:')
    print(f'  Total: {len(results)}')
    print(f'  Correctas: {matches}/{len(results)}')
    print(f'  Accuracy: {accuracy:.1f}%')
    print(f'  Score promedio: {avg_score:.2f}')
